<a href="https://colab.research.google.com/github/aninda-p/aninda-p/blob/main/Aninda_GenAI_Assignment1_Poblem1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# 1. Problem Statement

# A financial institution wants to predict whether a customer will default on a loan before approving it. Early identification of risky customers helps reduce financial loss.

# You are working as a Machine Learning Analyst and must build a classification model using the K-Nearest Neighbors (KNN) algorithm to predict loan default.

# This case introduces:

# Mixed feature types

# Financial risk interpretation

# Class imbalance awareness

import pandas as pd
from io import StringIO
import pprint

data = """
Age,Annual_Income,Credit_Score,Loan_Amount,Loan_Term,Employment_Type,Loan_Defaulter
28,6.5,720,5,5,Salaried,0
45,12,680,10,10,Self-Employed,1
35,8,750,6,7,Salaried,0
50,15,640,12,15,Self-Employed,1
30,7,710,5,5,Salaried,0
42,10,660,9,10,Salaried,1
26,5.5,730,4,4,Salaried,0
48,14,650,11,12,Self-Employed,1
38,9,700,7,8,Salaried,0
55,16,620,13,15,Self-Employed,1
"""

df = pd.read_csv(StringIO(data))
# df
pprint.pprint(df.to_dict(orient="list"), width=200)

# As KNN works with numeric values, I need to encode Employment_Type
from sklearn.preprocessing import LabelEncoder
label_encoder = LabelEncoder()
df["Employment_Type"] = label_encoder.fit_transform(df["Employment_Type"])
df

from sklearn.preprocessing import StandardScaler
# Features and target
# - X contains all the input variables the model will use to make predictions
# (Age, Income, Credit Score, Loan Amount, etc.)
# - y contains the output we want the model to learn

X = df.drop("Loan_Defaulter", axis=1)
y = df["Loan_Defaulter"]

# Scale features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Train
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.3, random_state=123) #Use the same random shuffle every time.

knc = KNeighborsClassifier(n_neighbors=3) # - K = 3, so the model looks at the 3 nearest neighbors to make a prediction.

knc.fit(X_train, y_train) # .fit trains the model on the training data.

y_pred = knc.predict(X_test) # Predict on the test set, gives predictions for the 3 test samples.
print(f'Classification Report: class 0[Not defaulter], calss 1[defaluter], support[test belonging to a particular class], precision[correctness], recall[model correctly found], F1[harmonic mean]')
print(classification_report(y_test, y_pred))

# Train a decision tree for comparision
from sklearn.tree import DecisionTreeClassifier

dt = DecisionTreeClassifier(max_depth=3, random_state=123)
dt.fit(X_train, y_train)

y_pred_dt = dt.predict(X_test)
print(classification_report(y_test, y_pred_dt))

# Interpretation

# Identify high-risk customers.
# Answer: Customers aged 45+, low credit score, self-employed are high risk customers
df["Predicted_Risk"] = knc.predict(X_scaled)
high_risk_customers = df[df["Predicted_Risk"] == 1]
print(f'High Risk Customers:')
print(high_risk_customers)

# What patterns lead to loan default?
# Answer: Patterns strongly associated with default
# - Low credit score (620–680)
# - High loan amount (10–13 lakhs)
# - Long loan term (10–15 years)
# - Self‑employed (income instability)
# - Older age group (45–55)

print(f'Loan Default Patterns:')
df.groupby("Loan_Defaulter").mean(numeric_only=True)

# How do credit scores and income influence predictions?
# Answer: credit scores and income influence features predict loan very accurately
# This should show loan defaulters have lower credit scores and income
df.groupby("Loan_Defaulter")[["Credit_Score", "Annual_Income"]].mean()


# Suggest banking policies based on model output.
# Answer: There consistent patterns among customers who default:
# low credit scores, high loan amounts, long loan terms, and self‑employment.
# Using these insights, a bank can design policies that reduce financial risk while still approving good customers.

# Compare KNN with Decision Trees for this problem.
# Answer: On comparing the prediction output of KNN and DecisionTree, it appears that the Decision Tree output is more accurate

# What happens if Loan Amount dominates distance calculation?
# Answer: KNN depends on customer and distance from features, If Loan Amount dominates
# the distance calculation, it breaks the model by ignoring other influential features,
# accuracy drops and predictions become unstable

# Should KNN be used in real-time loan approval systems?
# Answer: No, because of the following drawbacks
# Slow: as it needs to calculate distance from every customer in the database
# Poor accuracy: when compared to Decision Trees

{'Age': [28, 45, 35, 50, 30, 42, 26, 48, 38, 55],
 'Annual_Income': [6.5, 12.0, 8.0, 15.0, 7.0, 10.0, 5.5, 14.0, 9.0, 16.0],
 'Credit_Score': [720, 680, 750, 640, 710, 660, 730, 650, 700, 620],
 'Employment_Type': ['Salaried', 'Self-Employed', 'Salaried', 'Self-Employed', 'Salaried', 'Salaried', 'Salaried', 'Self-Employed', 'Salaried', 'Self-Employed'],
 'Loan_Amount': [5, 10, 6, 12, 5, 9, 4, 11, 7, 13],
 'Loan_Defaulter': [0, 1, 0, 1, 0, 1, 0, 1, 0, 1],
 'Loan_Term': [5, 10, 7, 15, 5, 10, 4, 12, 8, 15]}
Classification Report: class 0[Not defaulter], calss 1[defaluter], support[test belonging to a particular class], precision[correctness], recall[model correctly found], F1[harmonic mean]
              precision    recall  f1-score   support

           0       1.00      1.00      1.00         2
           1       1.00      1.00      1.00         1

    accuracy                           1.00         3
   macro avg       1.00      1.00      1.00         3
weighted avg       1.00      1.

,Credit_Score,Annual_Income
Loan_Defaulter,,
0,722.0,7.2
1,650.0,13.4
